In [1]:
import pandas as pd
import numpy as np

# 重塑分层索引

## stack和unstack 

In [2]:
arrays = ["bar", "bar", "baz", "baz"],["one", "two", "one", "two"]
index=pd.MultiIndex.from_arrays(arrays,names=["first", "second"])
frame=pd.DataFrame([[1,2],[3,4],[5,6],[7,8]], index=index, columns=["A", "B"])
frame

A  B
first second      
bar   one     1  2
      two     3  4
baz   one     5  6
      two     7  8

## stack：将列索引转换为行索引

In [3]:
stacked = frame.stack()
stacked 

first  second   
bar    one     A    1
               B    2
       two     A    3
               B    4
baz    one     A    5
               B    6
       two     A    7
               B    8
dtype: int64

## unstack：将行索引转换为列索引

In [4]:
display(stacked)
unnstack1 = stacked.unstack()
unnstack1

first  second   
bar    one     A    1
               B    2
       two     A    3
               B    4
baz    one     A    5
               B    6
       two     A    7
               B    8
dtype: int64

A  B
first second      
bar   one     1  2
      two     3  4
baz   one     5  6
      two     7  8

In [5]:
display(stacked)
unnstack2 = stacked.unstack('second')
unnstack2

first  second   
bar    one     A    1
               B    2
       two     A    3
               B    4
baz    one     A    5
               B    6
       two     A    7
               B    8
dtype: int64

second   one  two
first            
bar   A    1    3
      B    2    4
baz   A    5    7
      B    6    8

In [6]:
# 同时unstack两个级别
display(stacked)
result = stacked.unstack([1, 2])  
result

first  second   
bar    one     A    1
               B    2
       two     A    3
               B    4
baz    one     A    5
               B    6
       two     A    7
               B    8
dtype: int64

second one    two   
         A  B   A  B
first               
bar      1  2   3  4
baz      5  6   7  8

# 降采样

## resample

In [7]:
time_ser = pd.date_range('2022/10/01', periods=31)  #生成时间序列
stock_data = np.random.randint(40, 60, size=31)  #左闭右开
time_obj = pd.Series(stock_data, index=time_ser)
time_obj

2022-10-01    45
2022-10-02    41
2022-10-03    42
2022-10-04    40
2022-10-05    57
2022-10-06    58
2022-10-07    54
2022-10-08    52
2022-10-09    46
2022-10-10    54
2022-10-11    44
2022-10-12    59
2022-10-13    55
2022-10-14    55
2022-10-15    54
2022-10-16    50
2022-10-17    51
2022-10-18    56
2022-10-19    41
2022-10-20    48
2022-10-21    56
2022-10-22    43
2022-10-23    50
2022-10-24    52
2022-10-25    51
2022-10-26    55
2022-10-27    40
2022-10-28    55
2022-10-29    45
2022-10-30    49
2022-10-31    52
Freq: D, dtype: int32

In [8]:
# 每7天采集一次数据，实现降采样操作
result = time_obj.resample('7D').mean()
result.astype("int64")

2022-10-01    48
2022-10-08    52
2022-10-15    50
2022-10-22    49
2022-10-29    48
Freq: 7D, dtype: int64

In [9]:
result = time_obj.resample('W').mean()
result.astype("int64")

2022-10-02    43
2022-10-09    49
2022-10-16    53
2022-10-23    49
2022-10-30    49
2022-11-06    52
Freq: W-SUN, dtype: int64

## reample参数

In [10]:
w=pd.date_range(start = "2021/2/1", periods=10,freq="h")
y = pd.Series(np.arange(1,11),index=w)
y

2021-02-01 00:00:00     1
2021-02-01 01:00:00     2
2021-02-01 02:00:00     3
2021-02-01 03:00:00     4
2021-02-01 04:00:00     5
2021-02-01 05:00:00     6
2021-02-01 06:00:00     7
2021-02-01 07:00:00     8
2021-02-01 08:00:00     9
2021-02-01 09:00:00    10
Freq: h, dtype: int64

### close

closed：表示各时间段的哪一端是闭合的，可取值为'right'、'left'。

In [11]:
# closed="left"
# 左闭右开
y.resample("3h",closed="left").sum()  

2021-02-01 00:00:00     6
2021-02-01 03:00:00    15
2021-02-01 06:00:00    24
2021-02-01 09:00:00    10
Freq: 3h, dtype: int64

[00:00:00,03:00:00)

[03:00:00,06:00:00)

[06:00:00,09:00:00)

[09:00:00,结尾)

In [12]:
# closed="right"
# 左开右闭
y.resample("3h",closed="right").sum()  

2021-01-31 21:00:00     1
2021-02-01 00:00:00     9
2021-02-01 03:00:00    18
2021-02-01 06:00:00    27
Freq: 3h, dtype: int64

(21:00:00,00:00:00]

(00:00:00,03:00:00]

(03:00:00,06:00:00]

(06:00:00,09:00:00]

### label

label：表示降采样时设置的聚合结果的标签。

In [13]:
y

2021-02-01 00:00:00     1
2021-02-01 01:00:00     2
2021-02-01 02:00:00     3
2021-02-01 03:00:00     4
2021-02-01 04:00:00     5
2021-02-01 05:00:00     6
2021-02-01 06:00:00     7
2021-02-01 07:00:00     8
2021-02-01 08:00:00     9
2021-02-01 09:00:00    10
Freq: h, dtype: int64

In [14]:
# 左闭右开
y.resample("3h",closed="left",label="left").sum()

2021-02-01 00:00:00     6
2021-02-01 03:00:00    15
2021-02-01 06:00:00    24
2021-02-01 09:00:00    10
Freq: 3h, dtype: int64

[00:00:00,03:00:00)：取00：00：00

[03:00:00,06:00:00)：取03：00：00

[06:00:00,09:00:00)：取06：00：00

[09:00:00,结尾)：取09：00：00

In [15]:
# 左闭右开
y.resample("3h",closed="left",label="right").sum()

2021-02-01 03:00:00     6
2021-02-01 06:00:00    15
2021-02-01 09:00:00    24
2021-02-01 12:00:00    10
Freq: 3h, dtype: int64

[00:00:00,03:00:00)：取03：00：00

[03:00:00,06:00:00)：取06：00：00

[06:00:00,09:00:00)：取09：00：00

[09:00:00,12:00:00)：取12：00：00

# PCA降维

In [16]:
# # 安装所需要的包
# import sys
# !{sys.executable} -m pip install xlrd
# !{sys.executable} -m pip install openpyxl

In [17]:
import pandas as pd
 
#参数初始化
input_path = './data/principal_component.xls'
output_path = './data/dimention_reducted1.xlsx'  #降维后的数据保存路径
 
data = pd.read_excel(input_path, header = None) #读入数据，header = None没有列名
data  # 14行，8列（14个样本，每个样本有8个特征）

ImportError: `Import xlrd` failed. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install the xlrd package.

In [ ]:
# 第一次PCA（不降维，只是看看）
from sklearn.decomposition import PCA
pca = PCA()  # 创建一个PCA工具，没写n_components，所以不降维
pca.fit(data)  # 让PCA"学习"这组数据，找出主方向
print(pca.components_)  #返回模型的各个特征向量（这是8个新方向，每个方向有8个数字，代表怎么把原8列组合成新列）
print("---------------分割线-------------------")
print(pca.explained_variance_ratio_) #返回各个成分各自的方差百分比（只用前3个新列，就能保留原数据97%以上的信息）

[[ 0.56788461  0.2280431   0.23281436  0.22427336  0.3358618   0.43679539
   0.03861081  0.46466998]
 [ 0.64801531  0.24732373 -0.17085432 -0.2089819  -0.36050922 -0.55908747
   0.00186891  0.05910423]
 [-0.45139763  0.23802089 -0.17685792 -0.11843804 -0.05173347 -0.20091919
  -0.00124421  0.80699041]
 [-0.19404741  0.9021939  -0.00730164 -0.01424541  0.03106289  0.12563004
   0.11152105 -0.3448924 ]
 [-0.06133747 -0.03383817  0.12652433  0.64325682 -0.3896425  -0.10681901
   0.63233277  0.04720838]
 [ 0.02579655 -0.06678747  0.12816343 -0.57023937 -0.52642373  0.52280144
   0.31167833  0.0754221 ]
 [-0.03800378  0.09520111  0.15593386  0.34300352 -0.56640021  0.18985251
  -0.69902952  0.04505823]
 [-0.10147399  0.03937889  0.91023327 -0.18760016  0.06193777 -0.34598258
  -0.02090066  0.02137393]]
---------------分割线-------------------
[7.74011263e-01 1.56949443e-01 4.27594216e-02 2.40659228e-02
 1.50278048e-03 4.10990447e-04 2.07718405e-04 9.24594471e-05]


In [ ]:
#改变主成分个数（真正的降维）
pca=PCA(n_components=3)  #建立降为3维后的模型（告诉PCA：我要降到3列）
pca.fit(data)  # 训练数据（重新学习）
low_d=pca.transform(data) #把原始数据转换成3列的新数据
print(low_d) # 14行×3列

[[  8.19133694  16.90402785   3.90991029]
 [  0.28527403  -6.48074989  -4.62870368]
 [-23.70739074  -2.85245701  -0.4965231 ]
 [-14.43202637   2.29917325  -1.50272151]
 [  5.4304568   10.00704077   9.52086923]
 [ 24.15955898  -9.36428589   0.72657857]
 [ -3.66134607  -7.60198615  -2.36439873]
 [ 13.96761214  13.89123979  -6.44917778]
 [ 40.88093588 -13.25685287   4.16539368]
 [ -1.74887665  -4.23112299  -0.58980995]
 [-21.94321959  -2.36645883   1.33203832]
 [-36.70868069  -6.00536554   3.97183515]
 [  3.28750663   4.86380886   1.00424688]
 [  5.99885871   4.19398863  -8.59953736]]


In [ ]:
pd.DataFrame(low_d).to_excel(output_path)  #保存降维后的数据到本地
pca.inverse_transform(low_d)            #必要时可以用inverse_transform()函数来复原数据

array([[41.81945026, 17.92938537,  7.42743613,  6.38423781,  7.51911186,
         7.95581778,  1.89450158, 22.64634237],
       [26.03033486,  8.31048339, 11.0923029 , 10.50941053, 13.73592734,
        19.29219354,  1.55616178, 10.69991334],
       [12.8912027 ,  4.7200299 ,  4.15574756,  3.88084002,  4.15590258,
         5.95354081,  0.63142514,  3.10031979],
       [21.95107023,  7.86983692,  5.61296149,  5.00363184,  5.46598715,
         7.32692984,  1.00043437,  6.90279388],
       [33.2494621 , 16.9295226 ,  6.97070109,  6.54184048,  8.78799069,
         9.47854775,  1.76803069, 25.48379317],
       [35.30223656, 14.31635159, 16.19611986, 15.83211443, 22.51688172,
        30.25654088,  2.46591519, 25.94480913],
       [22.0404299 ,  7.67212745,  9.96458085,  9.59042702, 12.69748404,
        17.7402549 ,  1.39886681, 10.62704002],
       [47.82344306, 16.03581175, 11.11907058,  9.5362307 , 11.08119152,
        14.24461981,  2.12478649, 16.79265084],
       [40.72333307, 17.98533192